# 🚨 CrisisWorld — GRPO RL Training

**Model**: Qwen/Qwen2.5-0.5B-Instruct (default) or Meta-Llama-3-1B-Instruct  
**Algorithm**: GRPO (Group Relative Policy Optimization)  
**LoRA**: r=8, q_proj + v_proj  
**Exploration**: temperature=0.7, top_p=0.9, do_sample=True  

### Success Criteria
After training you should see in logs:
- `reward_std > 0`
- `loss > 0`
- `grad_norm > 0`
- rewards improving: `-80 → -40 → 10 → 60`

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected — switch runtime to GPU')

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
!git clone https://github.com/anilchowdary07/meta_x_scalar_ai_agents_city.git
%cd meta_x_scalar_ai_agents_city/backend

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q -r requirements-train.txt

In [ ]:
# ── 3. Set HuggingFace token (from Colab Secrets or paste here) ───────────────
import os

# Option A: Colab Secrets (recommended — never paste tokens in notebooks)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab Secrets')
except Exception:
    # Option B: Paste directly (remove before sharing)
    os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
    print('HF token set manually')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

In [ ]:
# ── 4. (Optional) Use smaller model if memory is tight ───────────────────────
# Default: Qwen/Qwen2.5-0.5B-Instruct (~1 GB)
# Uncomment for 1B Llama (needs ~4 GB VRAM):
# os.environ['CRISIS_MODEL'] = 'meta-llama/Meta-Llama-3-1B-Instruct'
print(f"Model: {os.getenv('CRISIS_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')}")

In [ ]:
# ── 5. Run 500-step training ──────────────────────────────────────────────────
# Features:
#   • max_steps=500 — significantly stronger policy
#   • resume_from_checkpoint=True — safe to re-run if Colab disconnects
#   • save every 50 steps (3 checkpoints kept)
#   • final model saved to checkpoints/crisisworld-grpo-final
#   • early-stop if reward_std < 1 after step 200 (prevents wasted GPU time)
#
# Expected log pattern:
#   Step 5   | Reward: -82.1 | Std: 38.7 | Loss: 0.087 | GradNorm: 1.24
#   Step 50  | Reward: -41.3 | Std: 44.2 | Loss: 0.062 | GradNorm: 0.98
#   Step 100 | Reward: -18.7 | Std: 51.0 | Loss: 0.051 | GradNorm: 0.87
#   Step 200 | Reward:  +8.4 | Std: 55.3 | Loss: 0.038 | GradNorm: 0.71
#   Step 500 | Reward: +42.1 | Std: 48.6 | Loss: 0.024 | GradNorm: 0.55

from train_grpo import train
train(
    output_dir="checkpoints/crisisworld-grpo-500",  # fresh dir — no conflict with old runs
    samples=128,       # reduce for faster Colab run
    max_steps=500,     # 500-step run
    horizon=10,        # episode length (shorter = faster reward eval)
    logging_steps=5,
    save_steps=50,     # checkpoint every 50 steps; last 3 kept
)

In [ ]:
# ── 6. Plot reward curve (500-step) ──────────────────────────────────────────
import json, numpy as np, matplotlib.pyplot as plt

# Try new 500-step dir, then final dir, then old fallback
for path in ['checkpoints/crisisworld-grpo-500-final/reward_curve.json',
             'checkpoints/crisisworld-grpo-500/reward_curve.json',
             'checkpoints/crisisworld-grpo-final/reward_curve.json']:
    try:
        with open(path) as f:
            data = json.load(f)
        print(f'Loaded from: {path}')
        break
    except FileNotFoundError:
        continue

rewards  = np.array(data['rewards'])
episodes = np.array(data['episodes'])

# 20-episode rolling average
window = min(20, len(rewards) // 4)
if window > 1:
    roll_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    roll_x   = episodes[window-1:]
else:
    roll_avg, roll_x = rewards, episodes

plt.figure(figsize=(12, 5))
plt.plot(episodes, rewards, color='#00ff8844', linewidth=1, label='Per-episode')
plt.plot(roll_x, roll_avg, color='#00ff88', linewidth=2.5, label=f'{window}-ep rolling avg')
plt.axhline(0, color='white', linestyle='--', alpha=0.25)
plt.title('CrisisWorld GRPO — 500-Step Reward Curve', color='white', fontsize=14, pad=12)
plt.xlabel('Episode', color='white'); plt.ylabel('Cumulative Reward', color='white')
plt.legend(facecolor='#0a1525', edgecolor='#334', labelcolor='white')
plt.gca().set_facecolor('#060d1a'); plt.gcf().set_facecolor('#060d1a')
plt.tick_params(colors='white')
plt.tight_layout()
out_path = path.replace('reward_curve.json', 'reward_curve.png')
plt.savefig(out_path, dpi=130)
plt.show()
print(f'Saved → {out_path}')
print(f'Episodes: {len(rewards)} | Final reward: {rewards[-1]:.1f} | '
      f'Peak: {rewards.max():.1f} | Mean last 50: {rewards[-50:].mean():.1f}')

In [ ]:
# ── 7. Upload weights to HuggingFace Hub ─────────────────────────────────────
# Replace with your HF username/repo
HF_REPO = 'anilchowdary07/crisisworld-grpo-qwen'

from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('checkpoints/crisisworld-grpo')
tokenizer = AutoTokenizer.from_pretrained('checkpoints/crisisworld-grpo')
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f'Model pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 8. (Optional) Push artifacts back to GitHub ───────────────────────────────
# !git config user.email 'you@example.com'
# !git config user.name 'Your Name'
# !git add checkpoints/
# !git commit -m 'feat: add trained GRPO weights and reward curve'
# !git push
print('Uncomment the lines above to push artifacts to GitHub.')